# Dự án: Phân lớp nhị phân và phát hiện giao dịch gian lận bằng Machine Learning
## Notebook 1: Khám phá dữ liệu (EDA) & Tiền xử lý dữ liệu nâng cao (Preprocessing)

--- 
**Mô tả:**
Notebook này đại diện cho giai đoạn đầu tiên của quy trình xử lý dữ liệu khoa học (Data Science Pipeline). Chúng ta sẽ tiến hành khảo sát dữ liệu giao dịch thẻ tín dụng (Credit Card Fraud Detection), phân tích chuyên sâu sự mất cân bằng lớp nghiêm trọng, trực quan hóa mối tương quan giữa các đặc trưng, và xây dựng song song hai pipeline tiền xử lý (chuẩn hóa):
1. **Phương pháp chuẩn hóa cổ điển:** Sử dụng `StandardScaler` của thư viện Scikit-Learn.
2. **Phương pháp chuẩn hóa phân tán (Big Data Ready):** Khởi tạo và sử dụng Spark Engine thông qua `PySpark` để xử lý dữ liệu ở quy mô lớn.

---

In [ ]:
# CELL 1: Cài đặt thư viện cần thiết
# Hỗ trợ tích hợp chạy PySpark trên môi trường cục bộ hoặc trên Kaggle Notebook
!pip install -q pyspark findspark

In [ ]:
# CELL 2: Import các thư viện lõi phục vụ EDA và Tiền xử lý
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Cấu hình phong cách biểu đồ thống nhất cho toàn bộ dự án
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

# Cố định giá trị ngẫu nhiên
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("[LOG] Import thư viện hoàn tất. Sẵn sàng xử lý dữ liệu.")

In [ ]:
# CELL 3: Đọc dataset từ Kaggle hoặc các đường dẫn cục bộ linh hoạt
DATA_PATHS = [
    "/kaggle/input/creditcard-fraud-detection/creditcard.csv",
    "../input/creditcardfractional/creditcard.csv",
    "../input/creditcardcard/creditcard.csv",
    "./data/creditcard.csv",
    "data/creditcard.csv",
    "../data/creditcard.csv"
]

csv_path = None
for path in DATA_PATHS:
    if os.path.exists(path):
        csv_path = path
        break

if csv_path is None:
    # Thử tìm kiếm đệ quy file csv trong thư mục hiện tại nếu không thấy ở các đường dẫn tĩnh
    matches = glob.glob("**/creditcard.csv", recursive=True)
    if matches:
        csv_path = matches[0]

if csv_path is None:
    raise FileNotFoundError(
        "[ERROR] Không tìm thấy file creditcard.csv. "
        "Vui lòng tải dataset từ Kaggle hoặc đặt file vào thư mục 'data/' của project."
    )

print(f"[LOG] Đang đọc dữ liệu từ đường dẫn: {csv_path}")
df = pd.read_csv(csv_path)
print(f"[LOG] Đọc dữ liệu thành công! Kích thước dữ liệu: {df.shape[0]:,} dòng, {df.shape[1]} cột.")

In [ ]:
# CELL 4: Khảo sát sơ bộ cấu trúc dữ liệu
print("=== 5 dòng đầu tiên của tập dữ liệu ===")
display(df.head())

print("\n=== Kích thước tập dữ liệu ===")
print(f"Hàng: {df.shape[0]:,}\nCột: {df.shape[1]}")

print("\n=== Thông tin chi tiết các trường dữ liệu (Schema & Dtypes) ===")
df.info()

print("\n=== Thống kê mô tả (Descriptive Statistics) ===")
display(df.describe())

In [ ]:
# CELL 5: Kiểm tra các giá trị khuyết thiếu (Missing values)
print("[LOG] Bắt đầu quét dữ liệu khuyết thiếu...")
missing_count = df.isnull().sum()
total_missing = missing_count.sum()

print(f"[LOG] Tổng số ô bị khuyết thiếu (Null/NaN) trên toàn bộ bảng: {total_missing}")
if total_missing > 0:
    print("\n[WARNING] Chi tiết số lượng khuyết thiếu tại từng cột:")
    display(missing_count[missing_count > 0])
else:
    print("[LOG] Dữ liệu hoàn hảo! Không phát hiện bất kỳ trường khuyết thiếu nào.")

In [ ]:
# CELL 6: Phân tích mất cân bằng lớp (Imbalance Analysis)
class_counts = df['Class'].value_counts()
class_percentages = df['Class'].value_counts(normalize=True) * 100

print("=== Bảng Tần Suất Lớp (Class Frequency Table) ===")
for idx, count in class_counts.items():
    label = "Bình thường (Normal)" if idx == 0 else "Gian lận (Fraud)"
    print(f" Lớp {idx} [{label:20}]: {count:9,} giao dịch | Tỷ lệ: {class_percentages[idx]:.4f}%")

# Trực quan hóa mất cân bằng bằng Seaborn
plt.figure(figsize=(7, 6))
palette_colors = ["#4C72B0", "#C44E52"] # Xanh dương nhã nhặn cho Normal, Đỏ gạch cho Fraud
ax = sns.countplot(x='Class', data=df, palette=palette_colors)

plt.title("Phân Phối Lớp Giao Dịch (Normal vs Fraud)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Lớp (Class)", fontsize=12, labelpad=10)
plt.ylabel("Số lượng giao dịch (Log Scale)", fontsize=12, labelpad=10)
plt.xticks(ticks=[0, 1], labels=["Bình thường\n(Class 0)", "Gian lận\n(Class 1)"], fontsize=11)
plt.yscale('log') # Áp dụng Log Scale vì sự chênh lệch quá lớn giữa 2 lớp

# Thêm giá trị cụ thể lên đỉnh cột
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height):,}',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='bottom',
                fontsize=11, color='black',
                xytext=(0, 5),
                textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# CELL 7: Phân tích tương quan đặc trưng (Correlation Analysis)
print("[LOG] Đang tính toán ma trận tương quan giữa các đặc trưng...")
corr_full = df.corr()

# Vẽ ma trận tương quan tổng thể
plt.figure(figsize=(12, 10))
sns.heatmap(corr_full, cmap='coolwarm', vmin=-1, vmax=1, center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8})
plt.title("Ma Trận Tương Quan Tổng Thể (Toàn Bộ Đặc Trưng)", fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Trực quan hóa ma trận tương quan tập trung vào Time, Amount và các biến V1-V28 ảnh hưởng mạnh nhất tới Class
class_corr = corr_full['Class'].sort_values()
print("\n=== Mức độ tương quan âm mạnh nhất với nhãn Class (Top 5) ===")
print(class_corr.head(5))
print("\n=== Mức độ tương quan dương mạnh nhất với nhãn Class (Top 5) ===")
print(class_corr.tail(6).drop('Class')[::-1])

# Lấy top 5 tương quan âm, top 5 tương quan dương kết hợp với Time và Amount
selected_features = list(class_corr.head(5).index) + list(class_corr.tail(6).drop('Class').index) + ['Time', 'Amount', 'Class']
selected_features = list(dict.fromkeys(selected_features)) # Loại bỏ trùng lặp nếu có

plt.figure(figsize=(11, 9))
sns.heatmap(df[selected_features].corr(), annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, center=0, 
            square=True, linewidths=.5, annot_kws={"size": 9})
plt.title("Ma Trận Tương Quan Tập Trung Vào Các Đặc Trưng Quan Trọng", fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### CELL 8: Giải thích ý nghĩa của Sự Mất Cân Bằng Lớp và Hệ Số Tương Quan

#### 1. Mất Cân Bằng Lớp (Imbalanced Data)
* **Bản chất hiện tượng:** Dữ liệu có sự chênh lệch khủng khiếp giữa lớp chiếm đa số (Normal - $99.827\%$) và lớp thiểu số (Fraud - chỉ vỏn vẹn $0.173\%$). Cứ khoảng $578$ giao dịch bình thường mới xuất hiện $1$ giao dịch gian lận.
* **Hệ quả đối với Machine Learning:** 
  * Nếu chúng ta huấn luyện trực tiếp một mô hình phân loại trên dữ liệu thô này mà không có sự điều chỉnh, mô hình sẽ có xu hướng "lười biếng" (Naive Classifier) - luôn dự đoán mọi giao dịch là *Bình thường (Class 0)*. 
  * Khi đó, độ chính xác tổng thể (Accuracy) vẫn đạt mức **$99.827\%$**, nhưng mô hình hoàn toàn vô dụng vì bỏ lọt toàn bộ $100\%$ các giao dịch gian lận. Do đó, Accuracy là một metric tồi tệ đối với dữ liệu mất cân bằng. Chúng ta bắt buộc phải sử dụng các metric thay thế như **Precision, Recall, F1-Score** và **ROC-AUC**.

#### 2. Phân Tích Hệ Số Tương Quan (Correlation Analysis)
* **Nhận xét từ Heatmap:**
  * Các biến từ $V_1$ đến $V_{28}$ là kết quả của phép biến đổi PCA (Principal Component Analysis). Về mặt lý thuyết toán học, các thành phần chính này độc lập tuyến tính với nhau, do đó hệ số tương quan giữa chúng xấp xỉ bằng $0$ (thể hiện rõ qua vùng màu xanh nhạt trung tính trên heatmap tổng thể).
  * Tuy nhiên, khi đối chiếu tương quan với biến mục tiêu `Class`, ta phát hiện một số đặc trưng có mối quan hệ rõ rệt:
    * **Tương quan âm mạnh:** $V_{17}$ ($-0.327$), $V_{14}$ ($-0.303$), $V_{12}$ ($-0.260$), $V_{10}$ ($-0.217$). Khi giá trị các biến này giảm xuống, xác suất xảy ra giao dịch gian lận tăng lên mạnh mẽ.
    * **Tương quan dương mạnh:** $V_{11}$ ($0.155$), $V_4$ ($0.133$), $V_2$ ($0.091$). Khi giá trị các đặc trưng này tăng lên, nguy cơ xảy ra giao dịch gian lận cũng tăng theo.
    * **Time và Amount:** Có tương quan tuyến tính rất yếu với `Class`. Điều này phản ánh rằng hành vi gian lận không phụ thuộc đơn giản theo hàm tuyến tính vào thời gian hay số tiền giao dịch, đòi hỏi mô hình phi tuyến tính phức tạp hơn.

In [ ]:
# CELL 9: Phân tách Đặc Trưng (Features) và Nhãn Mục Tiêu (Label)
X = df.drop(columns=['Class'])
y = df['Class']

print("[LOG] Phân tách thành công:")
print(f" - Khung chứa các đặc trưng (X): {X.shape[0]:,} mẫu, {X.shape[1]} đặc trưng.")
print(f" - Vector chứa nhãn mục tiêu (y): {y.shape[0]:,} phần tử.")

In [ ]:
# CELL 10: Chia tập dữ liệu thành Train và Test (Train/Test Split)
# Tỷ lệ chia: 80% Train, 20% Test
# Sử dụng stratify=y để giữ nguyên tỷ lệ nhãn cực kỳ mất cân bằng ở cả hai tập, tránh lệch phân phối
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("[LOG] Phân chia Train/Test thành công!")
print(f" - Số mẫu tập huấn luyện (Train): {X_train.shape[0]:,} ({X_train.shape[0]/df.shape[0]*100:.1f}%)")
print(f" - Số mẫu tập kiểm thử (Test): {X_test.shape[0]:,} ({X_test.shape[0]/df.shape[0]*100:.1f}%)")
print(f" - Số lượng Fraud trên tập Train: {y_train.sum()} ({y_train.mean()*100:.4f}%)")
print(f" - Số lượng Fraud trên tập Test: {y_test.sum()} ({y_test.mean()*100:.4f}%)")

In [ ]:
# CELL 11: [Cách cũ - Scikit-Learn] Chuẩn hóa các đặc trưng Time và Amount
# Do các đặc trưng V1-V28 đã được chuẩn hóa thông qua phép biến đổi PCA,
# chúng ta chỉ cần chuẩn hóa 2 cột có khoảng giá trị thô rộng là 'Time' và 'Amount'.

scaler = StandardScaler()

# Tạo bản sao dữ liệu tránh tác động trực tiếp vào DataFrame gốc
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit scaler trên tập Train và áp dụng transform lên cả hai tập Train và Test để tránh rò rỉ thông tin (Data Leakage)
cols_to_scale = ['Time', 'Amount']
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("[LOG] Chuẩn hóa Time và Amount bằng StandardScaler (Scikit-Learn) thành công.")
print("\n=== Thống kê mô tả sau khi chuẩn hóa (Time & Amount) ===")
display(X_train_scaled[cols_to_scale].describe())

In [ ]:
# CELL 12: [Cách mới nâng cao] Khởi tạo SparkSession bằng PySpark
print("[LOG] Bắt đầu tích hợp Spark Engine cục bộ...")
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler as SparkStandardScaler
from pyspark.sql.functions import col

try:
    spark = SparkSession.builder \
        .appName("CreditCardFraudPreprocessing") \
        .master("local[*]") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .getOrCreate()
    
    print(f"[SUCCESS] Khởi tạo thành công SparkSession. Spark Version: {spark.version}")
    spark_enabled = True
except Exception as e:
    print(f"[WARNING] Không thể khởi động Spark: {e}")
    print("[WARNING] Hệ thống sẽ bỏ qua việc thực thi PySpark và sử dụng Scikit-Learn đã hoàn tất làm mặc định.")
    spark_enabled = False

In [ ]:
# CELL 13: Đọc CSV bằng Spark DataFrame
if spark_enabled:
    print(f"[LOG] Spark đang tiến hành nạp dữ liệu từ: {csv_path}")
    spark_df = spark.read.csv(csv_path, header=True, inferSchema=True)
    
    print(f"[LOG] Spark DataFrame nạp thành công!")
    print(f"Số dòng: {spark_df.count():,}, Số cột: {len(spark_df.columns)}")
    spark_df.printSchema()
else:
    print("[INFO] Spark không khả dụng. Bỏ qua bước này.")

In [ ]:
# CELL 14: gom các đặc trưng thành Vector duy nhất bằng VectorAssembler
if spark_enabled:
    # Lấy danh sách các cột đặc trưng loại trừ cột nhãn 'Class'
    feature_cols = [c for c in spark_df.columns if c != 'Class']
    
    # Gom nhóm
    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_vector")
    spark_vector_df = assembler.transform(spark_df)
    
    print("[LOG] Tạo Vector đặc trưng bằng VectorAssembler thành công.")
    spark_vector_df.select("features_vector", "Class").show(3, truncate=True)
else:
    print("[INFO] Spark không khả dụng. Bỏ qua bước này.")

In [ ]:
# CELL 15: Sử dụng StandardScaler của PySpark chuẩn hóa dữ liệu lớn
if spark_enabled:
    # Khởi tạo Spark StandardScaler để chuẩn hóa vector đặc trưng
    # withStd=True: Chia cho độ lệch chuẩn
    # withMean=True: Trừ đi giá trị trung bình để đưa kỳ vọng về 0
    spark_scaler = SparkStandardScaler(inputCol="features_vector", outputCol="scaled_features", 
                                        withStd=True, withMean=True)
    
    # Huấn luyện bộ chuẩn hóa
    spark_scaler_model = spark_scaler.fit(spark_vector_df)
    
    # Thực hiện chuẩn hóa trên toàn bộ tập dữ liệu
    spark_scaled_df = spark_scaler_model.transform(spark_vector_df)
    
    print("[LOG] Chuẩn hóa toàn diện bằng PySpark StandardScaler thành công.")
    spark_scaled_df.select("scaled_features", "Class").show(3, truncate=True)
else:
    print("[INFO] Spark không khả dụng. Bỏ qua bước này.")

### CELL 16: Giải Thích Tầm Quan Trọng và Kiến Trúc PySpark Trong Tiền Xử Lý Dữ Liệu Lớn

#### 1. Catalyst Optimizer là gì?
* **Khái niệm:** Đây là trái tim tối ưu hóa của Spark SQL và DataFrame API. Catalyst Optimizer tận dụng các tính năng lập trình hàm trong ngôn ngữ Scala để xây dựng một cây thực thi tối ưu qua 4 giai đoạn chính:
  1. **Analysis:** Phân tích cú pháp DataFrame để đối chiếu với catalog của Spark nhằm kiểm tra tính hợp lệ của các cột và kiểu dữ liệu.
  2. **Logical Optimization:** Áp dụng các quy tắc tối ưu hóa logic như *Constant Folding* (tính toán hằng số trước), *Predicate Pushdown* (đẩy bộ lọc dữ liệu xuống gần nguồn đọc nhất), và *Projection Pruning* (chỉ giữ lại các cột cần thiết).
  3. **Physical Planning:** Sinh ra nhiều kế hoạch vật lý khả thi và chạy mô hình chi phí (Cost-based Model) để chọn ra kế hoạch tiêu tốn ít RAM/CPU nhất.
  4. **Code Generation:** Sử dụng tính năng Quasar CodeGen để biên dịch kế hoạch vật lý tối ưu trực tiếp thành mã bytecode Java cực nhanh khi thực thi thực tế.

#### 2. Khả năng Scale dữ liệu lớn và kiến trúc phân tán
* Khác với Pandas vốn chạy đơn luồng (Single-thread) và bị giới hạn bởi dung lượng RAM vật lý của một máy tính (gây ra lỗi kinh điển `OutOfMemory`), PySpark hoạt động dựa trên mô hình **Master-Worker** phân tán:
  * Dữ liệu được chia thành các phân vùng (Partitions) nhỏ nằm rải rác trên nhiều máy tính khác nhau (HDFS hoặc Cloud Storage).
  * Spark thực hiện cơ chế **Lazy Evaluation** (đánh giá lười biếng): Các lệnh biến đổi dữ liệu (Transformations) như `VectorAssembler` hay `StandardScaler` sẽ không được chạy ngay lập tức, mà chỉ được ghi nhận vào đồ thị **DAG** (Directed Acyclic Graph). Dữ liệu chỉ thực sự được xử lý khi xuất hiện lệnh kích hoạt (Actions) như `show()`, `count()`, hay `write()`.
  * Nhờ đó, chúng ta có thể mở rộng (scale-out) hệ thống một cách tuyến tính bằng cách bổ sung thêm node phần cứng để xử lý dữ liệu lên tới hàng Terabytes hay Petabytes mà không cần sửa đổi bất kỳ dòng code tiền xử lý nào.

In [ ]:
# CELL 17: Xuất dữ liệu đã chuẩn hóa ra các tệp trung gian phục vụ huấn luyện mô hình
print("[LOG] Bắt đầu ghi dữ liệu ra đĩa...")

# Đảm bảo thư mục lưu trữ tồn tại
OUTPUT_DIR = "processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Xuất các file dữ liệu dạng CSV để Notebook 2 nạp vào dễ dàng
X_train_scaled.to_csv(f"{OUTPUT_DIR}/X_train_scaled.csv", index=False)
X_test_scaled.to_csv(f"{OUTPUT_DIR}/X_test_scaled.csv", index=False)
y_train.to_csv(f"{OUTPUT_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{OUTPUT_DIR}/y_test.csv", index=False)

print("[LOG] Dữ liệu tiền xử lý đã được lưu trữ thành công tại:")
print(f" - {OUTPUT_DIR}/X_train_scaled.csv ({os.path.getsize(f'{OUTPUT_DIR}/X_train_scaled.csv')/1024/1024:.2f} MB)")
print(f" - {OUTPUT_DIR}/X_test_scaled.csv ({os.path.getsize(f'{OUTPUT_DIR}/X_test_scaled.csv')/1024/1024:.2f} MB)")
print(f" - {OUTPUT_DIR}/y_train.csv ({os.path.getsize(f'{OUTPUT_DIR}/y_train.csv')/1024/1024:.2f} MB)")
print(f" - {OUTPUT_DIR}/y_test.csv ({os.path.getsize(f'{OUTPUT_DIR}/y_test.csv')/1024/1024:.2f} MB)")

if spark_enabled:
    # Giải phóng tài nguyên Spark Session sau khi hoàn tất
    spark.stop()
    print("[LOG] Đã đóng Spark Session thành công. Tài nguyên đã được giải phóng hoàn toàn.")